This notebook is for the analysis of question 3 in the survey - how well explained is the response?

In [13]:
import logging
from datetime import datetime
from pathlib import Path

"""
This script loads in responses from classification csv file output by Zooniverse,
and normalizes them into a table with one row per task answered
"""

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from analysis_helpers import load_raw, wrap_labels

# PARAMETERS
classifications_csv_path = r'/4_survey_responses\may-survey-classifications_2025-06-17.csv'
output_dir = r'/4_survey_responses\analysis\may'
ml_responses_database_path = 'records_bedrock.db'
prompts_database_path = 'prompts.json'
subjects_path = r'/4_survey_responses\review-llm-responses-on-lca-tasks-subjects.csv'
workflow_id = 28845
exclude_prompts = [17]
exclude_models=['ollama_chat/deepseek-r1:8b'] # Deepseek-r1 was run with two inference platforms. We retain the results for the deepseek model with more parameters for a fair comparison

In [14]:
sns.set_theme(style="darkgrid")

# Load data

In [15]:
current_date = datetime.now().strftime('%Y%m%d')

logging.basicConfig(level=logging.INFO)

# Read classifications
classifications_df = pd.read_csv(classifications_csv_path)
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)


In [16]:
combined_final = load_raw(classifications_csv_path, subjects_path, prompts_database_path, exclude_models=exclude_models, exclude_prompt_ids=exclude_prompts, workflow_id=workflow_id)

FileNotFoundError: [Errno 2] No such file or directory: 'Y:\\projects\\unep-life-cycle-assessment-ml-paper-evaluation\\survey_responses\\review-llm-responses-on-lca-tasks-subjects.csv'

# Analyse Q3

In [ ]:
summary_q3 = combined_final.T3.value_counts()

In [ ]:
summary_q3

In [ ]:
# map t2 to numerical values
rating_description_to_number = {"3 - good": 3,
"2 - average": 2,
"1 - poor": 1,
"4 - expert / better than human": 4}

In [ ]:
combined_final['T3_score']=combined_final['T3'].apply(lambda x: rating_description_to_number[x])

In [ ]:
combined_final.columns

In [ ]:
# Bar Plot
plt.figure(figsize=(10, 6))
sns.barplot(x=summary_q3.index, y=summary_q3.values)
plt.title('Q2 - Explanation Quality of LLMs',fontsize=16)
plt.xlabel('Categories',fontsize=14)
plt.ylabel('Counts',fontsize=14)
plt.show()

In [ ]:
# Pie Chart with larger text
plt.figure(figsize=(8, 8))
plt.pie(summary_q3.values, labels=summary_q3.index, autopct='%1.1f%%', startangle=140, textprops={'fontsize': 14})
plt.title('Expert rating explanation quality of LLMs on LCA tasks (all models)', fontsize=16)
plt.show()

In [ ]:
q2_by_model = combined_final.groupby(['#llm_model', 'T3_score']).size().unstack(fill_value=0)

In [ ]:
q2_by_model

In [ ]:
model_T3_mean = combined_final.groupby('#llm_model')['T3_score'].mean().sort_values(ascending=False)
print(model_T3_mean)


In [ ]:



# Count Plot
plt.figure(figsize=(10, 6))
sns.barplot(x=model_T3_mean.index,y=model_T3_mean.values)
plt.title(f'Expert rating of explanation per model ({combined_final.shape[0]} reviews)')
plt.xlabel('Categories')
plt.ylabel('Rating')
plt.yticks(list(rating_description_to_number.values()), list(rating_description_to_number.keys()))


plt.subplots_adjust(left=0.15, bottom=0.3)
wrap_labels(plt.gca(), width=15)

plt.savefig(output_dir/f'{current_date}_q2_scientific_explanation_per_model.png')
plt.show()



# The same, but take the median

In [ ]:
model_T3_median = combined_final.groupby('#llm_model')['T3_score'].median().sort_values(ascending=False)
print(model_T3_mean)


In [ ]:

sns.set_theme(style="darkgrid")

# Count Plot
plt.figure(figsize=(10, 6))
ax = sns.barplot(x=model_T3_mean.index,y=model_T3_mean.values)
plt.title('Expert rating of explanation per model')
plt.xlabel('Categories')
plt.ylabel('Rating')
plt.yticks(list(rating_description_to_number.values()), list(rating_description_to_number.keys()))

# Add number of data points to each bar
for i, model in enumerate(combined_final['#llm_model'].unique()):
    count = combined_final[combined_final['#llm_model'] == model]['T3_score'].count()
    ax.text(i, ax.get_ylim()[0] + 0.1, f'n={count}', ha='center', va='bottom', fontsize=10)


plt.subplots_adjust(left=0.15, bottom=0.3)
wrap_labels(plt.gca(), width=15)

plt.savefig(output_dir/f'{current_date}_q2_scientific_explanation_per_model_median.png')
plt.show()


# Correlation between model parameter count and explanation score (Q3)

In [ ]:

from scipy import stats

# Create a mapping from model name to parameter count in billions (float)
# Attempt to parse from the model string (e.g., ':8b', '27b', '70b', '330b'). Case-insensitive.

def infer_params_b(model_name: str) -> float:
    if not isinstance(model_name, str):
        return float('nan')
    s = model_name.lower()
    # common pattern like ':8b' or '-r1:8b' or '330b'
    m = re.search(r'(\d+(?:\.\d+)?)\s*b', s)
    if m:
        try:
            return float(m.group(1))
        except ValueError:
            return float('nan')
    # sometimes tiny models encode like 'phi4' or 'llama3.1-8b' etc. If no 'b' found, return NaN
    return float('nan')

# Optional manual overrides for ambiguous names (extend as needed)
manual_params_b = {
    # Examples (uncomment/fill if needed):
    # 'ollama_chat/phi4:latest': 14.0,
}

def get_params_b(model_name: str) -> float:
    if model_name in manual_params_b:
        return manual_params_b[model_name]
    return infer_params_b(model_name)

# Build a per-model dataframe with mean, median, n, and params
model_stats = (
    combined_final
    .groupby('#llm_model')
    .agg(mean_T3=('T3_score', 'mean'), median_T3=('T3_score', 'median'), n=('T3_score', 'count'))
    .reset_index()
)
model_stats['params_b'] = model_stats['#llm_model'].map(get_params_b)

# Drop models without known parameter counts
model_stats_known = model_stats.dropna(subset=['params_b']).copy()

print('Per-model stats with params (first rows):')
print(model_stats_known.head())

# Compute correlations (per-model)
pearson_r, pearson_p = stats.pearsonr(model_stats_known['params_b'], model_stats_known['mean_T3']) if len(model_stats_known) > 1 else (float('nan'), float('nan'))
spearman_rho, spearman_p = stats.spearmanr(model_stats_known['params_b'], model_stats_known['mean_T3']) if len(model_stats_known) > 1 else (float('nan'), float('nan'))

print(f'Per-model Pearson r={pearson_r:.3f}, p={pearson_p:.3g}')
print(f'Per-model Spearman rho={spearman_rho:.3f}, p={spearman_p:.3g}')

# Scatter with regression line; point size by n
plt.figure(figsize=(8,6))
ax = sns.regplot(data=model_stats_known, x='params_b', y='mean_T3', scatter=True, ci=95, line_kws={'color':'red'})
# Scale point sizes: base 50 + 5 per datapoint (cap for readability)
sizes = (50 + model_stats_known['n'].clip(upper=100) * 5).values
plt.scatter(model_stats_known['params_b'], model_stats_known['mean_T3'], s=sizes, alpha=0.6, color='C0')

# Annotate each point with model short name (last path segment)
for _, row in model_stats_known.iterrows():
    short = row['#llm_model'].split('/')[-1]
    plt.annotate(short, (row['params_b'], row['mean_T3']), textcoords='offset points', xytext=(5,5), fontsize=9)

plt.title(f'Q3: Mean explanation score vs model params (per model)\nPearson r={pearson_r:.2f} (p={pearson_p:.2g}), Spearman rho={spearman_rho:.2f} (p={spearman_p:.2g})')
plt.xlabel('Parameters (Billions)')
plt.ylabel('Mean Q3 score')
plt.ylim(0.8, 4.2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / f'{current_date}_q3_params_vs_mean_score_per_model.png', dpi=200)
plt.show()


In [ ]:
# Per-response correlation (each response assigned model params)
combined_final['params_b'] = combined_final['#llm_model'].map(get_params_b)
per_response = combined_final.dropna(subset=['params_b']).copy()

if not per_response.empty:
    pr_pearson_r, pr_pearson_p = stats.pearsonr(per_response['params_b'], per_response['T3_score']) if len(per_response) > 1 else (float('nan'), float('nan'))
    pr_spearman_rho, pr_spearman_p = stats.spearmanr(per_response['params_b'], per_response['T3_score']) if len(per_response) > 1 else (float('nan'), float('nan'))

    print(f'Per-response Pearson r={pr_pearson_r:.3f}, p={pr_pearson_p:.3g}')
    print(f'Per-response Spearman rho={pr_spearman_rho:.3f}, p={pr_spearman_p:.3g}')

    plt.figure(figsize=(8,6))
    sns.regplot(data=per_response, x='params_b', y='T3_score', x_jitter=0.05, y_jitter=0.05, scatter_kws={'alpha':0.3, 's':20}, line_kws={'color':'red'})
    plt.title(f'Q3: Explanation score vs model params (per response)\nPearson r={pr_pearson_r:.2f} (p={pr_pearson_p:.2g}), Spearman rho={pr_spearman_rho:.2f} (p={pr_spearman_p:.2g})')
    plt.xlabel('Parameters (Billions)')
    plt.ylabel('Q3 score')
    plt.yticks(list(rating_description_to_number.values()), list(rating_description_to_number.keys()))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_dir / f'{current_date}_q3_params_vs_score_per_response.png', dpi=200)
    plt.show()
else:
    print('No per-response correlation computed: no responses with known parameter counts.')



# Violin plot

In [ ]:
# Add count of datapoints per model to combined_df
counts_per_model = combined_final.groupby('#llm_model').T3.agg(len).to_dict()

combined_final['data_points_per_model'] = combined_final['#llm_model'].apply(lambda x: counts_per_model[x])

In [ ]:
import numpy as np

def convert_dict_to_sorted_list(input_dict):
    # Sort the keys based on their values
    sorted_keys = sorted(input_dict, key=input_dict.get)
    return sorted_keys
ylabels_text = convert_dict_to_sorted_list(rating_description_to_number)
ylabels_text.insert(0,'')
# Violin Plot

plt.figure(figsize=(12, 6))

colors = sns.color_palette("light:#5A9", as_cmap=True)

ax = sns.swarmplot(x='#llm_model', y='T3_score', data=combined_final,order=list(model_T3_median.index),hue='data_points_per_model',palette=colors)
plt.title('Expert rating of explanation per model')
plt.xlabel('Categories')
plt.ylabel('Rating')
plt.yticks(np.arange(0,5,1), ylabels_text)

# Rotate x-tick labels and adjust alignment
plt.xticks(rotation=45, ha='right')
wrap_labels(plt.gca(), width=15)


# Add number of data points to each bar
for i, model in enumerate(combined_final['#llm_model'].unique()):
    count = combined_final[combined_final['#llm_model'] == model]['T3_score'].count()
    ax.text(i, ax.get_ylim()[0] + 0.1, f'n={count}', ha='center', va='bottom', fontsize=10)

ax.get_legend().remove()
plt.subplots_adjust(left=0.15, right=0.85)
plt.savefig(output_dir/f'{current_date}_q2_scientific_explanation_per_model_violin.png')

plt.show()

In [ ]:
import numpy as np

def convert_dict_to_sorted_list(input_dict):
    # Sort the keys based on their values
    sorted_keys = sorted(input_dict, key=input_dict.get)
    return sorted_keys

ylabels_text = convert_dict_to_sorted_list(rating_description_to_number)
ylabels_text.insert(0, '')

# Whisker Plot
plt.figure(figsize=(12, 6))

colors = sns.color_palette("light:#5A9", as_cmap=True)

ax = sns.violinplot(x='#llm_model', y='T3_score', data=combined_final, order=list(model_T3_median.index))
plt.title('Expert rating of explanation per model')
plt.xlabel('Categories')
plt.ylabel('Rating')
plt.yticks(np.arange(0, 5, 1), ylabels_text)

# Rotate x-tick labels and adjust alignment
plt.xticks(rotation=45, ha='right')
wrap_labels(plt.gca(), width=15)

# Add number of data points to each bar
for i, model in enumerate(combined_final['#llm_model'].unique()):
    count = combined_final[combined_final['#llm_model'] == model]['T3_score'].count()
    ax.text(i, ax.get_ylim()[1] - 0.1, f'n={count}', ha='center', va='top', fontsize=10)

plt.subplots_adjust(left=0.15, right=0.85)
plt.savefig(output_dir / f'{current_date}_q2_scientific_explanation_per_model_whisker.png')

plt.show()

In [ ]:
ylabels_text

# Analyse textual remarks in comment field for Q3

In [ ]:
combined_final.T8.drop_duplicates()

In [4]:

from wordcloud import  WordCloud
import multidict
import re

def get_frequency_dict_for_text(sentence):
    fullTermsDict = multidict.MultiDict()
    tmpDict = {}

    # making dict for counting frequencies
    for text in sentence.split(" "):
        if re.match("a|the|an|the|to|in|for|of|or|by|with|is|on|that|be|answer|response|but|not|it|\.|,", text):
            continue
        val = tmpDict.get(text, 0)
        tmpDict[text.lower()] = val + 1
    for key in tmpDict:
        fullTermsDict.add(key, tmpDict[key])
    return fullTermsDict



In [5]:
# Join all answers to get a single piece of text
all_answers_text=' '.join(combined_final.T8.drop_duplicates().to_list())

# Custom frequencies calculation
all_answers_frequencies = get_frequency_dict_for_text(all_answers_text)

# Create wordcloud
wordcloud_obj = WordCloud(background_color="white")
wordcloud_obj.generate_from_frequencies(all_answers_frequencies)

plt.grid(visible=False,which='both',axis='both')
plt.imshow(wordcloud_obj)
plt.show()

NameError: name 'combined_final' is not defined

In [6]:
plt.show()

In [7]:
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()
q3_extra_comments = combined_final.loc[:,'T8'].drop_duplicates()
q3_extra_comments = pd.DataFrame({'response_id':q3_extra_comments.index,'comment':q3_extra_comments.values})

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Artur\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


NameError: name 'combined_final' is not defined

In [ ]:
get_sentiment_nltk_vader = lambda x: sia.polarity_scores(x)['compound']
q3_extra_comments['sentiment_vader_simple'] = q3_extra_comments.comment.dropna().map(get_sentiment_nltk_vader)

In [ ]:
q3_extra_comments

In [ ]:
# Sentiment using FLAIR https://github.com/flairNLP/flair
# Flair gives better quality sentiment analyses on this type of dataset than VADER, as it is trained on longer-form movie reviews whereas NLTK's VADER is trained on short-form informal social media posts
from flair.data import Sentence
from flair.nn import Classifier

# load the NER tagger
tagger = Classifier.load('sentiment')


In [ ]:
def get_sentiment_flair(x, tagger):
    sentence = Sentence(x)
    tagger.predict(sentence)
    return pd.Series([sentence.get_label().value, sentence.get_label().score])
q3_extra_comments[['sentiment_flair_label','sentiment_flair_confidence']] = q3_extra_comments.comment.dropna().apply(get_sentiment_flair,tagger=tagger)

In [ ]:
q3_extra_comments

In [ ]:
sns. q3_extra_comments.sentiment_flair_label

In [ ]:
plt.figure(figsize=(8, 8))
sentiment_pivot = q3_extra_comments.sentiment_flair_label.value_counts()
plt.pie(sentiment_pivot, labels=sentiment_pivot.index, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 14})
plt.title(f'Q3: Explanation Quality: Sentiment Analysis ({q3_extra_comments.shape[0]} comments)')
plt.show()